In [ ]:
# Install correct stable versions
!pip uninstall -y langchain langchain-community langchain-core
!pip install langchain==0.1.17 langchain-community faiss-cpu transformers sentence-transformers pypdf

In [ ]:
# Upload multiple PDFs
from google.colab import files
uploaded = files.upload()

In [ ]:
# Create data folder and move PDFs into it
import os, shutil

os.makedirs("data", exist_ok=True)

for file in uploaded:
    shutil.move(file, "data/" + file)

print("✅ PDFs uploaded:", os.listdir("data"))

In [ ]:
# Load all PDF documents
from langchain_community.document_loaders import PyPDFLoader

documents = []

for file in os.listdir("data"):
    if file.endswith(".pdf"):
        loader = PyPDFLoader(f"data/{file}")
        documents.extend(loader.load())

print("✅ Documents loaded:", len(documents))

In [ ]:
# Split documents into chunks
from langchain.text_splitter import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,   # bigger chunks = faster
    chunk_overlap=100  # keeps context
)

chunks = splitter.split_documents(documents)

print("✅ Total chunks:", len(chunks))

In [ ]:
# Convert text into vector embeddings
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2"
)

print("✅ Embeddings ready")

In [ ]:
# Store embeddings in FAISS vector DB
from langchain_community.vectorstores import FAISS

db = FAISS.from_documents(chunks, embeddings)

# Save locally
db.save_local("faiss_index")

print("✅ Database created and saved")

In [ ]:
# Create retrieval + generation pipeline
from langchain.chains import RetrievalQA
from transformers import pipeline
from langchain_community.llms import HuggingFacePipeline

# Retriever (search top 3 chunks)
retriever = db.as_retriever(search_kwargs={"k": 3})

# Lightweight free model
pipe = pipeline("text-generation", model="gpt2")

llm = HuggingFacePipeline(pipeline=pipe)

# Combine retrieval + LLM
qa = RetrievalQA.from_chain_type(
    llm=llm,
    retriever=retriever,
    return_source_documents=True
)

print("✅ Chatbot ready")

In [ ]:
while True:
    query = input("Ask (type 'exit' to stop): ")

    if query.lower() == "exit":
        print("Stopped!")
        break

    result = qa(query)

    print("\nAnswer:", result["result"])

    print("\nSources:")
    for doc in result["source_documents"]:
        print(doc.metadata)